In [ ]:
# IMPORTS
import os, warnings, joblib, time, json, platform, itertools, sys
import numpy as np
import pandas as pd
import requests
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.linear_model import HuberRegressor, Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    precision_recall_fscore_support, average_precision_score
)
import xgboost as xgb
from xgboost import XGBRegressor
from IPython.display import display
from scipy.stats import wilcoxon
warnings.filterwarnings("ignore")
GATE_MODE = "fixed"

# CONFIG
OUT_DIR     = f"dghxh_ndbc_west_coast_{GATE_MODE}"
SPLIT_TIME  = pd.Timestamp("2021-06-01 00:00:00")
L           = 24
H_LIST      = [1, 3, 6, 12, 24]
TAU_LIST    = [0, 1, 2, 3, 4, 6]
VALID_FRAC  = 0.20
GRAPH_K     = 4
RUN_TUNED   = True
JSO_POP     = 6
JSO_ITERS   = 8
RANDOM_SEED = 42
SHUFFLE_SEED = 12345
TARGET_COL  = "wvht"
FEATURES    = ["wvht", "wtmp", "atmp", "pres", "u_wind", "v_wind"]

# Buoy coordinates
BUOY_COORDS = {
    "46011": (34.937, -120.999),
    "46014": (39.225, -123.980),
    "46025": (33.765, -119.077),
    "46026": (37.750, -122.838),
    "46042": (36.787, -122.408),
    "46047": (32.418, -119.535),
    "46050": (44.679, -124.535),
    "46053": (34.246, -119.842),
}
BUOY_IDS = list(BUOY_COORDS.keys())

# Date range for download
START_DATE = "2019-01-01T00:00:00Z"
END_DATE   = "2021-12-31T23:59:00Z"

# NDBC DATA DOWNLOAD
DOWNLOAD_YEARS = [2019, 2020, 2021]

def download_ndbc_year(buoy_id, year, raw_dir, max_retries=3):
    url = (f"https://www.ndbc.noaa.gov/data/historical/stdmet/"
           f"{buoy_id}h{year}.txt.gz")
    out_path = os.path.join(raw_dir, f"{buoy_id}h{year}.txt.gz")

    if os.path.exists(out_path) and os.path.getsize(out_path) > 1000:
        return out_path

    for attempt in range(max_retries):
        try:
            resp = requests.get(url, timeout=60)
            if resp.status_code == 200:
                with open(out_path, "wb") as f:
                    f.write(resp.content)
                return out_path
            else:
                print(f"    HTTP {resp.status_code} for {buoy_id}/{year}")
        except Exception as e:
            print(f"    Attempt {attempt+1} failed: {e}")
        time.sleep(2)
    return None

def parse_ndbc_file(gz_path, buoy_id):
    import gzip
    with gzip.open(gz_path, 'rt') as f:
        lines = f.readlines()
    header_line = lines[0].lstrip('#').strip()
    cols = header_line.split()

    skip = 1
    if lines[1].startswith('#'):
        skip = 2

    data_lines = [l for l in lines[skip:] if not l.startswith('#')]
    from io import StringIO
    df = pd.read_csv(StringIO(''.join(data_lines)),
                     sep=r'\s+', header=None, names=cols,
                     na_values=['99','999','9999','99.0','999.0','9999.0','MM'])

    yr_col = 'YY' if 'YY' in df.columns else 'YYYY'
    df[yr_col] = df[yr_col].astype(int)
    if df[yr_col].max() < 100:
        df[yr_col] = df[yr_col].apply(lambda y: y+2000 if y<50 else y+1900)

    mm_col = 'mm' if 'mm' in df.columns else None
    try:
        if mm_col:
            df['timestamp'] = pd.to_datetime(
                dict(year=df[yr_col], month=df['MM'], day=df['DD'],
                     hour=df['hh'], minute=df[mm_col]))
        else:
            df['timestamp'] = pd.to_datetime(
                dict(year=df[yr_col], month=df['MM'], day=df['DD'],
                     hour=df['hh']))
    except Exception:
        return None

    df = df.sort_values('timestamp').drop_duplicates('timestamp')

    col_map = {'WDIR':'wdir','WSPD':'wspd','WVHT':'wvht',
               'ATMP':'atmp','WTMP':'wtmp','PRES':'pres'}
    df = df.rename(columns=col_map)

    for col in ['wdir','wspd','wvht','atmp','wtmp','pres']:
        if col not in df.columns:
            df[col] = np.nan
        else:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    theta = np.radians(df['wdir'].fillna(0).to_numpy())
    spd   = df['wspd'].fillna(0).to_numpy()
    df['u_wind'] = (-np.sin(theta) * spd).astype(np.float32)
    df['v_wind'] = (-np.cos(theta) * spd).astype(np.float32)

    df['station_id'] = buoy_id
    df['lat'] = BUOY_COORDS[buoy_id][0]
    df['lon'] = BUOY_COORDS[buoy_id][1]

    return df[['timestamp','station_id','lat','lon',
               'wvht','wtmp','atmp','pres','u_wind','v_wind']]

def build_ndbc_dataset():
    os.makedirs(OUT_DIR, exist_ok=True)
    raw_dir    = os.path.join(OUT_DIR, "raw_gz")
    cache_path = os.path.join(OUT_DIR, "ndbc_raw_cache.parquet")
    os.makedirs(raw_dir, exist_ok=True)

    if os.path.exists(cache_path):
        print(f"Loading from cache: {cache_path}")
        return pd.read_parquet(cache_path)

    print("Downloading NDBC files")
    all_dfs = []
    for buoy_id in BUOY_IDS:
        buoy_dfs = []
        for year in DOWNLOAD_YEARS:
            print(f"  {buoy_id} {year}...", end=" ", flush=True)
            gz_path = download_ndbc_year(buoy_id, year, raw_dir)
            if gz_path is None:
                print("SKIP")
                continue
            parsed = parse_ndbc_file(gz_path, buoy_id)
            if parsed is None or len(parsed) == 0:
                print("PARSE ERROR")
                continue
            buoy_dfs.append(parsed)
            print(f"OK ({len(parsed)} rows)")

        if buoy_dfs:
            buoy_df = (pd.concat(buoy_dfs, ignore_index=True)
                         .sort_values("timestamp")
                         .drop_duplicates("timestamp")
                         .set_index("timestamp")
                         .resample("h")
                         .first()
                         .reset_index())
            buoy_df["station_id"] = buoy_id
            buoy_df["lat"] = BUOY_COORDS[buoy_id][0]
            buoy_df["lon"] = BUOY_COORDS[buoy_id][1]
            all_dfs.append(buoy_df)
            print(f"{buoy_id}: {len(buoy_df)} hourly rows total")
        else:
            print(f"{buoy_id}: no data, skipped")

    if not all_dfs:
        raise RuntimeError("no buoy data")

    df_all = (pd.concat(all_dfs, ignore_index=True)
                .sort_values(["timestamp","station_id"])
                .reset_index(drop=True))
    df_all.to_parquet(cache_path, index=False)
    print(f"\nSaved cache: {cache_path}")
    print(f"Total: {len(df_all):,} rows, {df_all.station_id.nunique()} buoys")
    print(f"Period: {df_all.timestamp.min()} to {df_all.timestamp.max()}")
    return df_all

# HELPER FUNCTIONS
def haversine_km_vec(lat1,lon1,lat2,lon2):
    R=6371.0; lat1,lon1,lat2,lon2=map(np.radians,[lat1,lon1,lat2,lon2])
    dlat=lat2-lat1; dlon=lon2-lon1
    a=np.sin(dlat/2)**2+np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2.0*R*np.arcsin(np.sqrt(a))

def bearing_radians(lat1,lon1,lat2,lon2):
    lat1,lon1,lat2,lon2=map(np.radians,[lat1,lon1,lat2,lon2]); dlon=lon2-lon1
    y=np.sin(dlon)*np.cos(lat2); x=np.cos(lat1)*np.sin(lat2)-np.sin(lat1)*np.cos(lat2)*np.cos(dlon)
    return np.arctan2(y,x)

def flatten_window_per_node(X):
    S,Lx,N,F=X.shape; return X.transpose(0,2,1,3).reshape(S*N,Lx*F)

def compute_tail_metrics(y_true,y_pred,percentiles=(90,95,99)):
    rows=[]
    for p in percentiles:
        thr=np.percentile(y_true,p); idx=y_true>=thr
        if idx.sum()==0: rows.append((p,thr,np.nan,np.nan,0))
        else: rows.append((p,thr,mean_absolute_error(y_true[idx],y_pred[idx]),
            np.sqrt(mean_squared_error(y_true[idx],y_pred[idx])),int(idx.sum())))
    return rows

def compute_mfb_nmse(yt,yp):
    yt=np.asarray(yt,dtype=float); yp=np.asarray(yp,dtype=float)
    return np.mean(2*(yp-yt)/(yp+yt+1e-8)), np.sum((yp-yt)**2)/(np.sum(yp*yt)+1e-8)

def build_fill_values_from_train_timeline(X):
    fv=np.nanmedian(X,axis=0); gf=np.nanmedian(X.reshape(-1,X.shape[-1]),axis=0)
    gf=np.where(np.isnan(gf),0,gf)
    for n in range(fv.shape[0]):
        for f in range(fv.shape[1]):
            if np.isnan(fv[n,f]): fv[n,f]=gf[f]
    return np.where(np.isnan(fv),0,fv).astype(np.float32)

def impute_windows(X,fv):
    X_imp=X.copy().astype(np.float32); S,Lx,N,F=X_imp.shape
    for s in range(S):
        for n in range(N):
            for f in range(F):
                ser=pd.Series(X_imp[s,:,n,f],dtype="float32").ffill().bfill()
                arr=ser.to_numpy(dtype=np.float32)
                if np.isnan(arr).any(): arr=np.where(np.isnan(arr),fv[n,f],arr)
                X_imp[s,:,n,f]=arr
    return X_imp

def build_nodes_from_coords(station_ids,coord_dict):
    return pd.DataFrame([{"station_id":sid,"lat":coord_dict[sid][0],
                           "lon":coord_dict[sid][1],"node_id":i}
                          for i,sid in enumerate(station_ids)])

def build_edges_from_nodes(nodes_df,k=4):
    coords=nodes_df[["lat","lon"]].to_numpy(dtype=float); N=len(coords)
    D=np.zeros((N,N))
    for i in range(N): D[i,:]=haversine_km_vec(coords[i,0],coords[i,1],coords[:,0],coords[:,1])
    sigma=np.median(D[D>0]); edges=[]
    for i in range(N):
        for j in np.argsort(D[i])[1:min(k+1,N)]:
            edges.append((i,j,float(np.exp(-(D[i,j]**2)/(2*sigma**2))),float(D[i,j])))
    return pd.DataFrame(edges,columns=["src","dst","w_dist","dist_km"])

def build_static_adj(nodes_df,k=4):
    coords=nodes_df[["lat","lon"]].to_numpy(dtype=float); N=len(coords)
    D=np.zeros((N,N))
    for i in range(N): D[i,:]=haversine_km_vec(coords[i,0],coords[i,1],coords[:,0],coords[:,1])
    sigma=np.median(D[D>0]); A=np.zeros((N,N),dtype=np.float32)
    for i in range(N):
        for j in np.argsort(D[i])[1:min(k+1,N)]:
            A[i,j]=np.exp(-(D[i,j]**2)/(2*sigma**2))
    rs=A.sum(axis=1,keepdims=True); return np.divide(A,rs,out=np.zeros_like(A),where=rs>0)

def wind_bearing_toward(u, v):
    return np.arctan2(u, v)


def build_dynamic_adj(u_t, v_t, ctx, alpha=4.0, eps=0.05, gate_mode=None):
    if gate_mode is None:
        gate_mode = globals().get("GATE_MODE", "fixed")
    src, dst = ctx["src"], ctx["dst"]

    if gate_mode == "legacy":
        theta = np.arctan2(v_t[src], u_t[src])
        bearing = ctx["edge_bearing"]
    elif gate_mode in ("fixed", "shuffled"):
        theta = wind_bearing_toward(u_t[dst], v_t[dst])
        key = "edge_bearing_shuffled" if gate_mode == "shuffled" else "edge_bearing"
        bearing = ctx[key] + np.pi
    else:
        raise ValueError("unknown gate_mode %r" % (gate_mode,))

    align = np.cos(theta - bearing)
    gate = eps + (1.0 - eps) / (1.0 + np.exp(-alpha * align))
    w_dyn = ctx["w_dist"] * gate
    A = np.zeros((ctx["N"], ctx["N"]), dtype=np.float32)
    A[src, dst] = w_dyn.astype(np.float32)
    rs = A.sum(axis=1, keepdims=True)
    return np.divide(A, rs, out=np.zeros_like(A), where=rs > 0)

def make_graph_features_dynamic(X, ctx, tau=0, alpha=4.0, eps=0.05, gate_mode=None):
    S, Lx, N, F = X.shape
    Z = np.zeros((S, N, 2 * F), dtype=np.float32)
    for s in range(S):
        x_now = X[s, -1]
        x_lag = X[s, -1 - tau] if tau > 0 else x_now
        A = build_dynamic_adj(x_now[:, ctx["u_idx"]], x_now[:, ctx["v_idx"]],
                              ctx, alpha=alpha, eps=eps, gate_mode=gate_mode)
        Z[s] = np.concatenate([x_now, A @ x_lag], axis=-1)
    return Z

def make_graph_features_static(X,A_static,tau=0):
    S,Lx,N,F=X.shape; Z=np.zeros((S,N,2*F),dtype=np.float32)
    for s in range(S):
        x_now=X[s,-1]; x_lag=X[s,-1-tau] if tau>0 else x_now
        Z[s]=np.concatenate([x_now,A_static@x_lag],axis=-1)
    return Z

def summarize_regression(yt,yp):
    return {"MAE":float(mean_absolute_error(yt,yp)),"RMSE":float(np.sqrt(mean_squared_error(yt,yp))),"R2":float(r2_score(yt,yp))}

def train_val_split_time_order(X,Y,M,times,frac=0.2):
    n=X.shape[0]; v=max(1,int(np.ceil(frac*n)))
    return ({"X":X[:n-v],"Y":Y[:n-v],"M":M[:n-v],"times":times[:n-v]},
            {"X":X[n-v:],"Y":Y[n-v:],"M":M[n-v:],"times":times[n-v:]})

def fit_flat_regressor(model,Xtr,Ytr,Mtr,Xte,Yte,Mte):
    Xt2=flatten_window_per_node(Xtr); Xe2=flatten_window_per_node(Xte)
    yt=Ytr.reshape(-1); ye=Yte.reshape(-1); mt=(Mtr.reshape(-1)>0.5); me=(Mte.reshape(-1)>0.5)
    model.fit(Xt2[mt],yt[mt]); return {"y_true":ye[me],"y_pred":model.predict(Xe2)[me],"model":model}

def fit_static_graph_regressor(model,Xtr,Ytr,Mtr,Xte,Yte,Mte,A_static,tau):
    Zt=make_graph_features_static(Xtr,A_static,tau); Ze=make_graph_features_static(Xte,A_static,tau)
    yt=Ytr.reshape(-1); ye=Yte.reshape(-1); mt=(Mtr.reshape(-1)>0.5); me=(Mte.reshape(-1)>0.5)
    Xt2=Zt.reshape(-1,Zt.shape[-1]); Xe2=Ze.reshape(-1,Ze.shape[-1])
    model.fit(Xt2[mt],yt[mt]); return {"y_true":ye[me],"y_pred":model.predict(Xe2)[me],"model":model}

def fit_huber_graph(Xtr,Ytr,Mtr,Xte,Yte,Mte,ctx,tau=0,alpha=4.0,eps=0.05,huber_alpha=1e-4, gate_mode=None):
    Zt=make_graph_features_dynamic(Xtr,ctx,tau,alpha,eps, gate_mode=gate_mode); Ze=make_graph_features_dynamic(Xte,ctx,tau,alpha,eps, gate_mode=gate_mode)
    yt=Ytr.reshape(-1); ye=Yte.reshape(-1); mt=(Mtr.reshape(-1)>0.5); me=(Mte.reshape(-1)>0.5)
    Xt2=Zt.reshape(-1,Zt.shape[-1]); Xe2=Ze.reshape(-1,Ze.shape[-1])
    huber=Pipeline([("sc",StandardScaler()),("h",HuberRegressor(epsilon=1.35,alpha=huber_alpha,max_iter=500))])
    huber.fit(Xt2[mt],yt[mt]); ptr=huber.predict(Xt2); pte=huber.predict(Xe2)
    return {"y_true":ye[me],"y_pred":pte[me],"pred_train_all":ptr,"pred_test_all":pte,
            "y_train_all":yt,"y_test_all":ye,"mask_train":mt,"mask_test":me,"model":huber}

def fit_huber_hybrid(Xtr,Ytr,Mtr,Xte,Yte,Mte,ctx,tau=0,alpha=4.0,eps=0.05,huber_alpha=1e-4,
                     xgb_params=None,num_boost_round=400,seed=42,w_mid=2.0,w_danger=5.0,w_tail=10.0, gate_mode=None):
    base=fit_huber_graph(Xtr,Ytr,Mtr,Xte,Yte,Mte,ctx,tau,alpha,eps,huber_alpha, gate_mode=gate_mode)
    yt=base["y_train_all"]; mt=base["mask_train"]; me=base["mask_test"]
    res=yt-base["pred_train_all"]
    Xg_tr=np.asarray(flatten_window_per_node(Xtr),dtype=np.float32)
    Xg_te=np.asarray(flatten_window_per_node(Xte),dtype=np.float32)
    t90,t95,t99=np.percentile(yt[mt],[90,95,99])
    w=np.ones_like(yt[mt],dtype=np.float32)
    w[yt[mt]>=t90]=w_mid; w[yt[mt]>=t95]=w_danger; w[yt[mt]>=t99]=w_tail
    dtrain=xgb.DMatrix(Xg_tr[mt],label=res[mt],weight=w)
    if xgb_params is None:
        xgb_params={"objective":"reg:pseudohubererror","max_depth":6,"eta":0.05,
                    "subsample":0.80,"colsample_bytree":0.80,"lambda":1.0,"tree_method":"hist","seed":seed,"verbosity":0}
    booster=xgb.train(xgb_params,dtrain,num_boost_round=int(num_boost_round))
    res_te=booster.predict(xgb.DMatrix(Xg_te)); final=base["pred_test_all"].copy(); final[me]=final[me]+res_te[me]
    return {"y_true":base["y_test_all"][me],"y_pred_graph":base["pred_test_all"][me],"y_pred_final":final[me],
            "gmodel":base["model"],"hmodel":booster}

def choose_best_tau_dynamic(tr,va,ctx,tau_list, gate_mode=None):
    best_tau,best_mae=None,np.inf; rows=[]
    for tau in tau_list:
        if tau>=tr["X"].shape[1]: continue
        res=fit_huber_graph(tr["X"],tr["Y"],tr["M"],va["X"],va["Y"],va["M"],ctx,tau=tau, gate_mode=gate_mode)
        mae=mean_absolute_error(res["y_true"],res["y_pred"]); rows.append({"tau":tau,"val_MAE":mae})
        if mae<best_mae: best_mae=mae; best_tau=tau
    return best_tau, pd.DataFrame(rows)

def choose_best_tau_static(tr,va,A_static,tau_list):
    best_tau,best_mae=None,np.inf; rows=[]
    for tau in tau_list:
        if tau>=tr["X"].shape[1]: continue
        m=Pipeline([("sc",StandardScaler()),("h",HuberRegressor(epsilon=1.35,max_iter=500))])
        res=fit_static_graph_regressor(m,tr["X"],tr["Y"],tr["M"],va["X"],va["Y"],va["M"],A_static,tau)
        mae=mean_absolute_error(res["y_true"],res["y_pred"]); rows.append({"tau":tau,"val_MAE":mae})
        if mae<best_mae: best_mae=mae; best_tau=tau
    return best_tau, pd.DataFrame(rows)

def eval_hybrid_params(params,tau,tr,va,ctx):
    al,ep,lha,md,eta,sub,col,lam,nr,wm,wd_w,wt=params
    hybrid=fit_huber_hybrid(tr["X"],tr["Y"],tr["M"],va["X"],va["Y"],va["M"],ctx,tau=tau,
        alpha=float(al),eps=float(ep),huber_alpha=10**float(lha),
        xgb_params={"objective":"reg:pseudohubererror","max_depth":int(round(md)),"eta":float(eta),
            "subsample":float(sub),"colsample_bytree":float(col),"lambda":float(lam),
            "tree_method":"hist","seed":RANDOM_SEED,"verbosity":0},
        num_boost_round=int(round(nr)),seed=RANDOM_SEED,w_mid=float(wm),w_danger=float(wd_w),w_tail=float(wt), gate_mode=GATE_MODE)
    mae=mean_absolute_error(hybrid["y_true"],hybrid["y_pred_final"])
    idx=hybrid["y_true"]>=np.percentile(hybrid["y_true"],99)
    if idx.sum()>0:
        mfb,_=compute_mfb_nmse(hybrid["y_true"][idx],hybrid["y_pred_final"][idx])
        return mae+10*abs(mfb)
    return mae

def jso_optimize(tr,va,ctx,tau,bounds,n_pop=12,iters=15,seed=1):
    np.random.seed(seed); dim=bounds.shape[0]
    pop=bounds[:,0]+np.random.rand(n_pop,dim)*(bounds[:,1]-bounds[:,0])
    fit=np.array([eval_hybrid_params(p,tau,tr,va,ctx) for p in pop])
    best_p=pop[np.argmin(fit)].copy(); best_f=float(fit.min())
    for it in range(iters):
        c=1-it/max(iters,1); new_pop=pop.copy()
        for i in range(n_pop):
            if np.random.rand()<0.5:
                cand=pop[i]+np.random.randn(dim)*c*0.1+c*(best_p-pop[i])*np.random.rand(dim)
            else:
                j=np.random.randint(0,n_pop); cand=pop[i]+(pop[j]-pop[i])*(np.random.rand(dim)-0.5)*c
            new_pop[i]=np.clip(cand,bounds[:,0],bounds[:,1])
        nf=np.array([eval_hybrid_params(p,tau,tr,va,ctx) for p in new_pop])
        better=nf<fit; pop[better]=new_pop[better]; fit[better]=nf[better]
        if fit.min()<best_f: best_f=float(fit.min()); best_p=pop[np.argmin(fit)].copy()
        print(f"  iter {it+1:02d}/{iters} best={best_f:.4f}")
    return best_p, best_f

def add_result_row(rows,tail_rows,H,split,model,y_true,y_pred,extra=None):
    s=summarize_regression(y_true,y_pred)
    row={"H":H,"split":split,"model":model,"MAE":s["MAE"],"RMSE":s["RMSE"],"R2":s["R2"],"n_test":len(y_true)}
    if extra: row.update(extra)
    rows.append(row)
    for p,thr,mae,rmse,n_tail in compute_tail_metrics(y_true,y_pred):
        tr={"H":H,"split":split,"model":model,"percentile":p,"threshold":float(thr),
            "tail_MAE":float(mae) if pd.notna(mae) else np.nan,
            "tail_RMSE":float(rmse) if pd.notna(rmse) else np.nan,"n_tail":int(n_tail)}
        if extra: tr.update(extra)
        tail_rows.append(tr)

# BUILD TENSOR
os.makedirs(OUT_DIR, exist_ok=True)
df = build_ndbc_dataset()
station_ids = [bid for bid in BUOY_IDS if bid in df["station_id"].unique()]
N = len(station_ids)
print(f"Active buoys: {N}  ({station_ids})")
assert N >= 5

u_idx = FEATURES.index("u_wind"); v_idx = FEATURES.index("v_wind")

all_times = pd.date_range(df["timestamp"].min(), df["timestamp"].max(), freq="h")
base = pd.MultiIndex.from_product([all_times, station_ids],
                                   names=["timestamp","station_id"]).to_frame(index=False)
aligned = base.merge(df[["timestamp","station_id"]+FEATURES],
                     on=["timestamp","station_id"], how="left")

X_feat = []
for feat in FEATURES:
    mat = aligned.pivot(index="timestamp", columns="station_id",
                        values=feat).reindex(all_times)[station_ids]
    X_feat.append(mat.to_numpy(dtype=np.float32))
X_all_raw = np.stack(X_feat, axis=-1)

target_raw = aligned.pivot(index="timestamp", columns="station_id",
                            values=TARGET_COL).reindex(all_times)[station_ids].to_numpy(dtype=np.float32)
Y_mask_full = (~np.isnan(target_raw)).astype(np.float32)
fill_values = build_fill_values_from_train_timeline(X_all_raw[all_times < SPLIT_TIME])

print(f"Tensor {X_all_raw.shape}  WVHT coverage: {Y_mask_full.mean()*100:.1f}%")
print(f"WVHT range: {np.nanmin(target_raw):.2f} - {np.nanmax(target_raw):.2f} m")
print(f"WVHT 99th pct: {np.nanpercentile(target_raw,99):.2f} m")

# BUILD GRAPH
nodes    = build_nodes_from_coords(station_ids, BUOY_COORDS)
edges_df = build_edges_from_nodes(nodes, k=min(GRAPH_K, N-1))
A_static = build_static_adj(nodes, k=min(GRAPH_K, N-1))

src = edges_df["src"].to_numpy(dtype=int)
dst = edges_df["dst"].to_numpy(dtype=int)
w_dist = edges_df["w_dist"].to_numpy(dtype=np.float32)
edge_bearing = bearing_radians(
    nodes.loc[src,"lat"].to_numpy(), nodes.loc[src,"lon"].to_numpy(),
    nodes.loc[dst,"lat"].to_numpy(), nodes.loc[dst,"lon"].to_numpy())

edge_bearing_shuffled = edge_bearing[
    np.random.RandomState(SHUFFLE_SEED).permutation(len(edge_bearing))]

graph_ctx = {"src":src,"dst":dst,"w_dist":w_dist,"edge_bearing":edge_bearing,
             "edge_bearing_shuffled": edge_bearing_shuffled,
             "u_idx":u_idx,"v_idx":v_idx,"N":N}

# WINDOWING
def build_windows(X_raw,Y_raw,Y_mask,all_times,split_time,L,H_list):
    n_times=len(all_times); data={}
    for H in H_list:
        X_wins,Y_wins,M_wins,ttimes=[],[],[],[]
        for t in range(L, n_times-H):
            X_wins.append(X_raw[t-L:t]); Y_wins.append(Y_raw[t+H])
            M_wins.append(Y_mask[t+H]); ttimes.append(all_times[t+H])
        X_arr=np.stack(X_wins); Y_arr=np.stack(Y_wins)
        M_arr=np.stack(M_wins); t_arr=np.array(ttimes)
        test_mask=t_arr>=split_time; train_mask=~test_mask
        data[H]={"X_train_imp":impute_windows(X_arr[train_mask],fill_values),
                 "X_test_imp": impute_windows(X_arr[test_mask], fill_values),
                 "Y_train":Y_arr[train_mask],"Y_test":Y_arr[test_mask],
                 "M_train":M_arr[train_mask],"M_test":M_arr[test_mask],
                 "target_times_train":t_arr[train_mask],"target_times_test":t_arr[test_mask]}
        print(f"H={H}h  train={train_mask.sum()}  test={test_mask.sum()}")
    return data

print("Building windows...")
all_data = build_windows(X_all_raw, target_raw, Y_mask_full, all_times, SPLIT_TIME, L, H_LIST)

bounds = np.array([[1.0,8.0],[0.01,0.20],[-5.0,-2.0],[3.0,8.0],[0.02,0.15],
                   [0.60,1.00],[0.60,1.00],[0.10,5.0],[200.0,600.0],
                   [1.5,4.0],[3.0,8.0],[6.0,20.0]])

# MAIN TRAINING LOOP
# CONFIG
SEEDS      = [42, 43, 44]
SEED_MODE  = "refit"
BASE_OUT   = OUT_DIR
PERCENTILES = (90, 95, 99)
ALPHA      = 0.05
MIN_MODEL_BYTES = 100_000

def _chk(path, min_bytes=1000):
    sz = os.path.getsize(path)
    assert sz >= min_bytes
    print(f"    {os.path.basename(path):28s} {sz/1e6:9.3f} MB")

def _masked_index(M):
    S, N = M.shape
    si, ni = np.meshgrid(np.arange(S), np.arange(N), indexing="ij")
    m = M.reshape(-1) > 0.5
    return si.reshape(-1)[m], ni.reshape(-1)[m]

FROZEN = {}
seed_summaries = []

for SEED in SEEDS:
    t_seed = time.time()
    np.random.seed(SEED)
    try:
        import random as _random; _random.seed(SEED)
    except Exception:
        pass
    RANDOM_SEED = SEED
    OUT_DIR = f"{BASE_OUT}_seed{SEED}" if len(SEEDS) > 1 else BASE_OUT
    os.makedirs(OUT_DIR, exist_ok=True)
    print(f"\n{'#'*70}\n# SEED {SEED}   (mode={SEED_MODE})  : {OUT_DIR}\n{'#'*70}")

    results_rows, tail_rows, tau_rows, hp_rows = [], [], [], []
    all_models, all_preds = {}, {}

    for H in H_LIST:
        t_h = time.time()
        print(f"\n{'='*55}\nHORIZON H={H}h\n{'='*55}")
        pack = all_data[H]
        Xtr, Ytr, Mtr = pack["X_train_imp"], pack["Y_train"], pack["M_train"]
        Xte, Yte, Mte = pack["X_test_imp"],  pack["Y_test"],  pack["M_test"]

        tr, va = train_val_split_time_order(
            Xtr, Ytr, Mtr, pack["target_times_train"], frac=VALID_FRAC)

        if SEED_MODE == "refit" and H in FROZEN:
            best_tau_dyn    = FROZEN[H]["tau_dyn"]
            best_tau_static = FROZEN[H]["tau_static"]
            print(f"tau (frozen) dyn={best_tau_dyn} static={best_tau_static}")
        else:
            best_tau_dyn,    tt_d = choose_best_tau_dynamic(tr, va, graph_ctx, TAU_LIST)
            best_tau_static, tt_s = choose_best_tau_static(tr, va, A_static, TAU_LIST)
            tau_rows.extend(tt_d.assign(H=H, family="dynamic").to_dict("records"))
            tau_rows.extend(tt_s.assign(H=H, family="static").to_dict("records"))
            print(f"tau_dyn={best_tau_dyn}  tau_static={best_tau_static}")

        all_models[H] = {"tau_dyn": best_tau_dyn, "tau_static": best_tau_static}
        all_preds[H]  = {}

        def _record(name, res, key="y_pred", model_obj=None, extra=None):
            yp = res[key]
            add_result_row(results_rows, tail_rows, H, "test", name,
                           res["y_true"], yp, extra=extra or {})
            all_preds[H][name] = np.asarray(yp, dtype=np.float32)
            if model_obj is not None:
                all_models[H][name] = model_obj
            print(f"  {name} done")

        for name, model in [
            ("Ridge", Pipeline([("sc", StandardScaler()), ("r", Ridge(alpha=10.0))])),
            ("Huber", Pipeline([("sc", StandardScaler()),
                                ("h", HuberRegressor(epsilon=1.35, max_iter=1000))])),
            ("RF",    RandomForestRegressor(n_estimators=100, max_depth=8,
                                            n_jobs=-1, random_state=SEED)),
            ("XGB",   XGBRegressor(n_estimators=200, max_depth=8, learning_rate=0.07,
                                   subsample=0.9, colsample_bytree=0.8,
                                   tree_method="hist", n_jobs=-1, random_state=SEED)),
        ]:
            res = fit_flat_regressor(model, Xtr, Ytr, Mtr, Xte, Yte, Mte)
            _record(name, res, model_obj=res.get("model", model))

        for name, model in [
            ("SG-Ridge", Pipeline([("sc", StandardScaler()), ("r", Ridge(alpha=10.0))])),
            ("SG-Huber", Pipeline([("sc", StandardScaler()),
                                   ("h", HuberRegressor(epsilon=1.35, max_iter=1000))])),
        ]:
            res = fit_static_graph_regressor(model, Xtr, Ytr, Mtr, Xte, Yte, Mte,
                                             A_static=A_static, tau=best_tau_static)
            _record(name, res, model_obj=res.get("model", model),
                    extra={"tau": best_tau_static})

        res = fit_huber_graph(Xtr, Ytr, Mtr, Xte, Yte, Mte, graph_ctx, tau=best_tau_dyn)
        _record("Huber-Graph", res, model_obj=res["model"],
                extra={"tau": best_tau_dyn})

        res = fit_huber_hybrid(Xtr, Ytr, Mtr, Xte, Yte, Mte, graph_ctx,
                               tau=best_tau_dyn, num_boost_round=400, seed=SEED)
        _record("Hybrid", res, key="y_pred_final",
                model_obj=(res.get("graph_model"), res.get("resid_model")),
                extra={"tau": best_tau_dyn})
        all_models[H]["hybrid_graph"] = (res.get("graph_model"), res.get("resid_model"))

        if RUN_TUNED:
            if SEED_MODE == "refit" and H in FROZEN:
                best_p   = FROZEN[H]["best_p"]
                best_obj = FROZEN[H]["best_obj"]
                print("  JSO (frozen from first seed)")
            else:
                best_p, best_obj = jso_optimize(tr, va, graph_ctx, best_tau_dyn,
                                                bounds, JSO_POP, JSO_ITERS, SEED)
                FROZEN.setdefault(H, {}).update(
                    dict(best_p=best_p, best_obj=best_obj,
                         tau_dyn=best_tau_dyn, tau_static=best_tau_static))
            (al, ep, lha, md, eta, sub, col, lam, nr, wm, wd_w, wt) = best_p
            ha = 10 ** float(lha)

            hp_rows.append(dict(H=H, seed=SEED, tau_dyn=best_tau_dyn,
                tau_static=best_tau_static, val_obj=float(best_obj),
                alpha=float(al), eps=float(ep), huber_alpha=ha,
                max_depth=int(round(md)), eta=float(eta), subsample=float(sub),
                colsample_bytree=float(col), lam=float(lam),
                num_boost_round=int(round(nr)), w_mid=float(wm),
                w_danger=float(wd_w), w_tail=float(wt)))

            res_tg = fit_huber_graph(Xtr, Ytr, Mtr, Xte, Yte, Mte, graph_ctx,
                                     tau=best_tau_dyn, alpha=float(al),
                                     eps=float(ep), huber_alpha=ha)
            _record("Tuned-Huber-Graph", res_tg, model_obj=res_tg["model"],
                    extra={"tau": best_tau_dyn, "val_obj": float(best_obj)})
            all_models[H]["tuned_graph"] = res_tg["model"]

            res_th = fit_huber_hybrid(
                Xtr, Ytr, Mtr, Xte, Yte, Mte, graph_ctx, tau=best_tau_dyn,
                alpha=float(al), eps=float(ep), huber_alpha=ha,
                xgb_params={"objective": "reg:pseudohubererror",
                            "max_depth": int(round(md)), "eta": float(eta),
                            "subsample": float(sub), "colsample_bytree": float(col),
                            "lambda": float(lam), "tree_method": "hist",
                            "seed": SEED, "verbosity": 0},
                num_boost_round=int(round(nr)), seed=SEED,
                w_mid=float(wm), w_danger=float(wd_w), w_tail=float(wt))
            _record("Tuned-Hybrid", res_th, key="y_pred_final",
                    extra={"tau": best_tau_dyn, "val_obj": float(best_obj)})
            all_models[H]["tuned_hybrid_graph"] = (res_th.get("graph_model"),
                                                   res_th.get("resid_model"))

        _mte = Mte.reshape(-1) > 0.5
        _yt  = Yte.reshape(-1)[_mte]
        _tix = target_idx if "target_idx" in dir() else 0
        _L   = Xte.shape[1]
        per  = Xte[:, -1, :, _tix].reshape(-1)[_mte]
        sn   = (Xte[:, H, :, _tix].reshape(-1)[_mte] if H < _L else per)
        for nm, pv in [("Persistence", per), ("SeasonalNaive", sn)]:
            add_result_row(results_rows, tail_rows, H, "test", nm, _yt, pv)
            all_preds[H][nm] = np.asarray(pv, dtype=np.float32)
        print(f"  Persistence / SeasonalNaive done")
        print(f"  H={H}h time: {time.time()-t_h:.1f}s")

    print(f"\n{'='*60}\nSAVING (seed {SEED})\n{'='*60}")
    p = os.path.join(OUT_DIR, "all_models.pkl")
    joblib.dump(all_models, p); _chk(p, MIN_MODEL_BYTES)

    pd.DataFrame(results_rows).to_csv(os.path.join(OUT_DIR, "results_main.csv"), index=False)
    pd.DataFrame(tail_rows).to_csv(os.path.join(OUT_DIR, "results_tail.csv"), index=False)
    if tau_rows: pd.DataFrame(tau_rows).to_csv(os.path.join(OUT_DIR, "tau_search.csv"), index=False)
    if hp_rows:  pd.DataFrame(hp_rows).to_csv(os.path.join(OUT_DIR, "hyperparams.csv"), index=False)
    print(f"    results_main.csv / results_tail.csv / hyperparams.csv")
    for H in H_LIST:
        d = os.path.join(OUT_DIR, f"H{H}"); os.makedirs(d, exist_ok=True)
        pack = all_data[H]
        for k in ["X_train_imp", "X_test_imp", "Y_train", "Y_test", "M_train", "M_test"]:
            if k in pack: np.save(os.path.join(d, f"{k}.npy"), pack[k])
        si, ni = _masked_index(pack["M_test"])
        tt = np.asarray(pack["target_times_test"])
        store = {"y_true": pack["Y_test"].reshape(-1)[pack["M_test"].reshape(-1) > 0.5],
                 "sample_idx": si, "station_idx": ni,
                 "target_time": tt[si].astype("datetime64[s]").astype("int64")}
        store.update({f"pred__{k}": v for k, v in all_preds[H].items()})
        np.savez_compressed(os.path.join(d, "predictions.npz"), **store)
        _chk(os.path.join(d, "predictions.npz"), 1000)

    try:
        df_raw.to_parquet(os.path.join(OUT_DIR, "raw_cache.parquet"), index=False)
        _chk(os.path.join(OUT_DIR, "raw_cache.parquet"), 10_000)
    except NameError:
        pass

    json.dump({"saved_at": time.strftime("%Y-%m-%dT%H:%M:%S"), "seed": SEED,
               "seed_mode": SEED_MODE, "H_LIST": list(H_LIST),
               "python": platform.python_version(), "numpy": np.__version__,
               "pandas": pd.__version__,
               "split_time": str(SPLIT_TIME) if "SPLIT_TIME" in dir() else None,
               "target_col": TARGET_COL if "TARGET_COL" in dir() else None,
               "L": int(all_data[H_LIST[0]]["X_test_imp"].shape[1]),
               "n_stations": int(all_data[H_LIST[0]]["X_test_imp"].shape[2]),
               "n_features": int(all_data[H_LIST[0]]["X_test_imp"].shape[3]),
               "models": sorted(all_preds[H_LIST[0]].keys()),
               "wall_seconds": round(time.time() - t_seed, 1)},
              open(os.path.join(OUT_DIR, "run_manifest.json"), "w"), indent=2)
    print("    run_manifest.json")
    seed_summaries.append((SEED, OUT_DIR))

# SIGNIFICANCE + SEED POOLING
print(f"\n{'#'*70}\n# SIGNIFICANCE\n{'#'*70}")

def _eff(ea, eb):
    d = ea - eb; nz = d[d != 0]
    wf = float((d < 0).mean()); md = float(np.median(d))
    if len(nz) == 0: return wf, md, 0.0
    r = np.argsort(np.argsort(np.abs(nz))) + 1
    return wf, md, float((r[nz < 0].sum() - r[nz > 0].sum()) / r.sum())

sig_rows = []
for SEED, sdir in seed_summaries:
    for H in H_LIST:
        z = np.load(os.path.join(sdir, f"H{H}", "predictions.npz"))
        yt = z["y_true"]
        preds = {k[6:]: z[k] for k in z.files if k.startswith("pred__")}
        for pct in PERCENTILES:
            m = yt >= np.percentile(yt, pct)
            for a, b in itertools.permutations(preds, 2):
                ea, eb = np.abs(yt[m]-preds[a][m]), np.abs(yt[m]-preds[b][m])
                if len(ea) < 10 or np.all(ea == eb): continue
                wf, md, rbc = _eff(ea, eb)
                sig_rows.append(dict(seed=SEED, H=H, percentile=pct,
                    candidate=a, comparator=b, n_tail=int(m.sum()),
                    MAE_candidate=float(ea.mean()), MAE_comparator=float(eb.mean()),
                    delta_pct=float((ea.mean()/eb.mean()-1)*100),
                    median_paired_diff=md, win_fraction=wf, rank_biserial=rbc,
                    p_one_sided=wilcoxon(ea, eb, alternative="less").pvalue,
                    p_two_sided=wilcoxon(ea, eb, alternative="two-sided").pvalue))

sig = pd.DataFrame(sig_rows)
sig.to_csv(os.path.join(BASE_OUT, "significance_all.csv"), index=False)
bonf = ALPHA / max(len(sig), 1)
print(f"{len(sig)} tests across {len(seed_summaries)} seeds; "
      f"Bonferroni threshold = {bonf:.2e}")

KEY = [("Tuned-Hybrid", "XGB"), ("Tuned-Hybrid", "Persistence"),
       ("Tuned-Hybrid", "Hybrid"), ("XGB", "Persistence")]
for pct in PERCENTILES:
    print(f"\n{'='*104}\np{pct} TAIL  (mean ± sd over seeds)")
    for a, b in KEY:
        sub = sig[(sig.percentile == pct) & (sig.candidate == a) & (sig.comparator == b)]
        if sub.empty: continue
        for H in H_LIST:
            r = sub[sub.H == H]
            if r.empty: continue
            sig_frac = float((r.p_one_sided < bonf).mean())
            print(f"  H={H:2d} {a:>16s} vs {b:<13s} "
                  f"d={r.delta_pct.mean():+6.2f}±{r.delta_pct.std(ddof=0):5.2f}%  "
                  f"win={r.win_fraction.mean():.2f}  rbc={r.rank_biserial.mean():+.2f}  "
                  f"sig in {sig_frac*100:3.0f}% of seeds")

pooled = (sig.groupby(["H","percentile","candidate","comparator"])
          .agg(delta_mean=("delta_pct","mean"), delta_sd=("delta_pct","std"),
               win_mean=("win_fraction","mean"), rbc_mean=("rank_biserial","mean"),
               n_seeds=("seed","nunique")).reset_index())
pooled.to_csv(os.path.join(BASE_OUT, "significance_pooled.csv"), index=False)
print(f"\nsaved: {BASE_OUT}/significance_all.csv and significance_pooled.csv")

  46011 2019... OK (16789 rows)
  46011 2020... OK (52253 rows)
  46011 2021... OK (51559 rows)
  46011: 26305 hourly rows total
  46014 2019... OK (52130 rows)
  46014 2020... OK (52203 rows)
  46014 2021... OK (51509 rows)
  46014: 26304 hourly rows total
  46025 2019... OK (52126 rows)
  46025 2020... OK (52203 rows)
  46025 2021... OK (51542 rows)
  46025: 26304 hourly rows total
  46026 2019... OK (52143 rows)
  46026 2020... OK (52239 rows)
  46026 2021... OK (51563 rows)
  46026: 26304 hourly rows total
  46042 2019... OK (14855 rows)
  46042 2020... OK (52260 rows)
  46042 2021... OK (51454 rows)
  46042: 26304 hourly rows total
  46047 2019... OK (35520 rows)
  46047 2020... OK (52132 rows)
  46047 2021... OK (51494 rows)
  46047: 26304 hourly rows total
  46050 2019... OK (52055 rows)
  46050 2020... OK (52211 rows)
  46050 2021... OK (51561 rows)
  46050: 26304 hourly rows total
  46053 2019... OK (52135 rows)
  46053 2020... OK (52230 rows)
  46053 2021... OK (51515 rows)
 